# Mérida Urban Intelligence: Geospatial Data Warehouse Analytics

This notebook demonstrates the end-to-end analytical workflow for the **Mérida Urban Intelligence Geospatial Data Warehouse**:
1. **Geographic Assessment:** Spatial unit evaluation (**Urban Blocks / Manzanas** vs. **AGEBs**).
2. **Geocoding & Spatial Snapping:** Mapping incident coordinates to the 15,728 urban blocks of Mérida.
3. **Territorial KPIs:** SQL-equivalent queries for incident typologies, density, and commercial diversity.
4. **Inferential Spatial Analysis:** Statistical correlations (Pearson/Spearman), Global Moran's $I$, and Local Moran's $I$ (LISA) cluster detection.

In [ ]:
import os
import sys
from pathlib import Path

# Ensure project root is in sys.path
root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root_dir))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

print("Project Root:", root_dir)

## 1. Inspecting the Geocoded Public Safety Layer

We load the processed incident facts (`fact_traffic_incidents.csv`) and accident typologies (`dim_incident_type.csv`).

In [ ]:
df_incidents = pd.read_csv(root_dir / 'data' / 'processed' / 'fact_traffic_incidents.csv')
df_types = pd.read_csv(root_dir / 'data' / 'processed' / 'dim_incident_type.csv')

print(f"Total geocoded incidents: {len(df_incidents):,}")
display(df_types)
display(df_incidents.head(5))

## 2. Block-Level Breakdown: Accident Volume and Predominant Typology

Emulating the PostGIS analytical view `view_block_incident_kpis`, we compute block-level aggregates and identify the predominant accident type per block.

In [ ]:
# Join with typology dimension
df_merged = df_incidents.merge(df_types, on='tipo_id', how='left')

# Aggregate by block (cvegeo_manzana)
block_kpis = df_merged.groupby('cvegeo_manzana').agg(
    total_incidentes=('incident_id', 'count'),
    choques_moto=('tipo_accidente', lambda s: (s == 'Colisión con motocicleta').sum()),
    atropellamientos=('tipo_accidente', lambda s: (s == 'Colisión con peatón').sum()),
    choques_vehiculo=('tipo_accidente', lambda s: (s == 'Colisión con vehículo').sum()),
    accidentes_con_alcohol=('aliento_alcohol', 'sum'),
    tipo_predominante=('tipo_accidente', lambda s: s.mode()[0] if not s.empty else 'N/A')
).reset_index()

print("Top 10 Blocks with Highest Incident Concentration:")
display(block_kpis.sort_values('total_incidentes', ascending=False).head(10))

## 3. Spatial Visualizations: Density & Typology Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 9))

img_density = Image.open(root_dir / 'outputs' / 'maps' / 'merida_incident_density_blocks.png')
axes[0].imshow(img_density)
axes[0].axis('off')
axes[0].set_title('Spatial Incident Density per Block', fontsize=12, fontweight='bold')

img_types = Image.open(root_dir / 'outputs' / 'maps' / 'merida_accident_types_predominant.png')
axes[1].imshow(img_types)
axes[1].axis('off')
axes[1].set_title('Territorial Distribution by Accident Typology', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Inferential Spatial Analysis: Global Moran's $I$ & LISA Cluster Map

- **Global Moran's $I$:** Evaluates global spatial autocorrelation against spatial randomness ($I > 0$ indicates spatial clustering).
- **LISA (Local Moran's $I$):** Deconstructs global autocorrelation into local spatial quadrants (**High-High** hotspots, **Low-Low** coldspots, and spatial outliers).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

img_moran = Image.open(root_dir / 'outputs' / 'figures' / 'moran_scatterplot_accidents.png')
axes[0].imshow(img_moran)
axes[0].axis('off')
axes[0].set_title('Moran Scatterplot (Global Moran\'s I)', fontsize=12, fontweight='bold')

img_lisa = Image.open(root_dir / 'outputs' / 'maps' / 'lisa_clusters_accidents.png')
axes[1].imshow(img_lisa)
axes[1].axis('off')
axes[1].set_title('LISA Cluster Map (Local Indicators of Spatial Association)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Statistical Correlations Across Territorial Indicators

In [ ]:
plt.figure(figsize=(10, 7))
img_corr = Image.open(root_dir / 'outputs' / 'figures' / 'correlation_matrix.png')
plt.imshow(img_corr)
plt.axis('off')
plt.title('Spearman Rank Correlation Matrix', fontsize=12, fontweight='bold')
plt.show()